In [ ]:
from glob import glob

from IPython.display import display, clear_output

from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib.patches import Polygon as MplPolygon

from rasterio.windows import from_bounds

from sarpy.visualization.remap import density, brighter

from shapely.geometry import Polygon
from shapely.geometry import box

import json
import os

import geopandas as gpd
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio as rs
import shapely
import skimage

### Read Olmo prediction raster

In [ ]:
img_rs_pred = rs.open(f'/Users/juliansanders/Downloads/data 2/datasets/olmo/defid2/oerun_cluster_30d_trim/pred_for_slides/results/results_raster/result_epsg32633_0.tif')
img_pred = img_rs_pred.read(1)


### Read ground truth label and sentinel-2 image raster

In [ ]:
img_rs_gt_s2 = rs.open(f'/Users/juliansanders/Downloads/data 2/datasets/olmo/defid2/oerun_cluster_30d_trim/pred_for_slides/dataset_0/windows/group_partition_0/partition_0_e37f3d8a581b288ff047/layers/sentinel2_l2a_sep2019/B05_B06_B07_B8A_B11_B12/geotiff.tif')
img_rs_gt = rs.open(f'/Users/juliansanders/Downloads/label.tif')



img_gt = img_rs_gt.read(1)
img_s2 = img_rs_gt_s2.read([3,2,1]).transpose((1,2,0))
img_gt.shape

### Get starting position to align the pred/gt

In [ ]:
starting_pos = img_rs_pred.index(*img_rs_gt.xy(0,0))
starting_pos

In [ ]:
# clip to extent
img_pred_clip = img_pred[starting_pos[0]:starting_pos[0]+img_gt.shape[0],
                         starting_pos[1]:starting_pos[1]+img_gt.shape[1]]

### Plot the ground truth labels and Olmo Earth base predictions


In [ ]:
plt.figure(figsize=(20,10))
plt.subplot(1,3,1)
plt.title('S2 ortho resampled to 3m GSD')
plt.imshow(brighter(img_s2))
plt.subplot(1,3,2)
plt.title('Truth label')
plt.imshow(np.ma.array(img_gt, mask=img_gt==255))
plt.subplot(1,3,3)
plt.title('OLMOEARTH_V1_BASE Pred')
plt.imshow(np.ma.array(img_pred_clip, mask=img_gt==255))
plt.show()

In [ ]:
from sarpy.visualization.remap import density, brighter
from skimage.exposure import equalize_hist

This cell compares the distribution of raw Sentinel‑2 pixel intensities with the distribution after histogram equalization. The blue histogram shows the original normalized values, while the red histogram reveals how equalization spreads the intensities to enhance contrast.

In [ ]:
plt.hist(img_s2.ravel()/img_s2.max(), bins=100)
plt.hist(equalize_hist(img_s2).ravel(), bins=100, color='red')
plt.show()

Overlay the ground truth and model prediction masks on top of the Sentinel-2 RGB image for visual comparison. The masks are resized to match the Sentinel-2 image dimensions, and contours are drawn with blue solid lines for ground truth and yellow dashed lines for predictions.

In [ ]:
from skimage.transform import resize
from matplotlib.lines import Line2D

plt.figure(figsize=(10,10))
plt.title('Overlay: Ground Truth vs Prediction on Sentinel-2')


# Show the base Sentinel-2 satellite image
plt.imshow(brighter(img_s2))

# Resize masks so they match the Sentinel-2 image dimensions
gt_resized = resize(
    (img_gt == 1).astype(float),
    img_s2.shape[:2],
    order=0,                # nearest neighbor
    preserve_range=True,
    anti_aliasing=False
)

pred_resized = resize(
    (img_pred_clip == 1).astype(float),
    img_s2.shape[:2],
    order=0,
    preserve_range=True,
    anti_aliasing=False
)

# Plot contours
plt.contour(
    gt_resized,
    colors='blue',
    linewidths=1.5,
    linestyles='solid'
)

plt.contour(
    pred_resized,
    colors='yellow',
    linewidths=1.5,
    linestyles='dashed'
)

# Create legend
legend_elements = [
    Line2D([0], [0], color='blue', lw=2, linestyle='solid', label='Ground Truth'),
    Line2D([0], [0], color='yellow', lw=2, linestyle='dashed', label='Prediction')
]

plt.legend(handles=legend_elements, loc='upper right')

# Remove axis for cleaner visualization
plt.axis('off')

plt.show()

This cell extracts polygons from the ground‑truth and prediction masks, rescales them to match the Sentinel‑2 image resolution, and classifies each as a true positive, false positive, or false negative based on overlap. It then overlays only the polygon outlines—colored green, red, or blue—on top of the original image for clear visual comparison.

In [ ]:
from skimage import measure

# Convert binary mask to shapely polygons
def mask_to_polygons(mask):
    polys = []
    contours = measure.find_contours(mask, 0.5)

    for c in contours:
        poly = Polygon([(p[1], p[0]) for p in c])  # (x,y)

        if poly.is_valid and poly.area > 1:
            polys.append(poly)

    return polys


# Create polygons in original mask coordinate system
gt_polys = mask_to_polygons(img_gt == 1)
pred_polys = mask_to_polygons(img_pred_clip == 1)


# Classify polygons
TP = []
FP = []
FN = []

for p in pred_polys:
    if any(p.intersects(g) for g in gt_polys):
        TP.append(p)
    else:
        FP.append(p)

for g in gt_polys:
    if not any(g.intersects(p) for p in pred_polys):
        FN.append(g)


# Compute scaling for visualization on Sentinel-2
scale_y = img_s2.shape[0] / img_gt.shape[0]
scale_x = img_s2.shape[1] / img_gt.shape[1]


def scale_poly(poly):
    x, y = poly.exterior.xy
    x = np.array(x) * scale_x
    y = np.array(y) * scale_y
    return np.vstack([x, y]).T

# Visualization
plt.figure(figsize=(12,12))
plt.imshow(brighter(img_s2))
ax = plt.gca()


def draw_polys(polys, color):
    for poly in polys:
        coords = scale_poly(poly)
        ax.add_patch(MplPolygon(
            coords,
            fill=False,
            edgecolor=color,
            linewidth=2
        ))


# Green = predictions overlapping GT
draw_polys(TP, 'green')

# Red = predictions with no GT overlap
draw_polys(FP, 'red')

# Blue = GT with no prediction overlap
draw_polys(FN, 'blue')


plt.axis('off')
plt.title("Green = TP, Red = FP, Blue = FN")
plt.show()

This cell computes pixel‑level true positives, false positives, and false negatives from the ground‑truth and prediction masks, assigns each a distinct RGB color, and resizes the result to match the Sentinel‑2 image resolution. It then blends the colored error map with the normalized satellite image to visualize TP (green), FP (red), and FN (blue) directly on the base image.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from skimage.transform import resize
from skimage.exposure import equalize_hist

# Binary masks
gt = img_gt.copy()
pred = img_pred_clip.copy()

valid_mask = gt != 255 

gt_bin = (gt == 1)
pred_bin = (pred == 1)

TP = (gt_bin & pred_bin) & valid_mask
FP = (~gt_bin & pred_bin) & valid_mask
FN = (gt_bin & ~pred_bin) & valid_mask

# Create RGB overlay mask
overlay = np.zeros((*gt.shape, 3), dtype=np.float32)

overlay[TP] = [0, 1, 0]     # green
overlay[FP] = [1, 0, 0]     # red
overlay[FN] = [0, 0, 1]     # blue

# Normalize S2 image
s2_norm = equalize_hist(img_s2.astype(np.float32))
s2_norm = (s2_norm - s2_norm.min()) / (s2_norm.max() - s2_norm.min())

# Resize overlay to match S2 image
overlay_resized = resize(
    overlay,
    s2_norm.shape,
    order=0,             
    preserve_range=True,
    anti_aliasing=False
)

# Alpha blend
alpha = 0.5
blended = (1 - alpha) * s2_norm + alpha * overlay_resized

#Plot
plt.figure(figsize=(12, 12))
plt.imshow(blended)
plt.title("Sentinel‑2 with TP (green), FP (red), FN (blue)")
plt.axis("off")
plt.show()

This cell loads multiple time‑sequenced Sentinel‑2 band groups, normalizes each image, and blends them with the TP/FP/FN overlay to visualize label quality across months. Interactive widgets allow you to switch band groups, scroll through dates, and adjust overlay transparency for rapid comparison of temporal imagery.

In [ ]:
# Initialize specific band paths for your underlying image
band_groups = {
    "B05_B06_B07_B8A_B11_B12": [
        "sentinel2_l2a_mar2019/B05_B06_B07_B8A_B11_B12/geotiff.tif",
        "sentinel2_l2a_apr2019/B05_B06_B07_B8A_B11_B12/geotiff.tif",
        "sentinel2_l2a_may2019/B05_B06_B07_B8A_B11_B12/geotiff.tif",
        "sentinel2_l2a_june2019/B05_B06_B07_B8A_B11_B12/geotiff.tif",
        "sentinel2_l2a_july2019/B05_B06_B07_B8A_B11_B12/geotiff.tif",
    ],
    "B01_B09": [
        "sentinel2_l2a_mar2019/B01_B09/geotiff.tif",
        "sentinel2_l2a_apr2019/B01_B09/geotiff.tif",
        "sentinel2_l2a_may2019/B01_B09/geotiff.tif",
        "sentinel2_l2a_june2019/B01_B09/geotiff.tif",
        "sentinel2_l2a_july2019/B01_B09/geotiff.tif",
    ],
    "B02_B03_B04_B08": [
        "sentinel2_l2a_mar2019/B02_B03_B04_B08/geotiff.tif",
        "sentinel2_l2a_apr2019/B02_B03_B04_B08/geotiff.tif",
        "sentinel2_l2a_may2019/B02_B03_B04_B08/geotiff.tif",
        "sentinel2_l2a_june2019/B02_B03_B04_B08/geotiff.tif",
        "sentinel2_l2a_july2019/B02_B03_B04_B08/geotiff.tif",
    ]
}

# Initialize the base path for your data
base_path = "/Users/juliansanders/Downloads/data 2/datasets/olmo/defid2/oerun_cluster_30d_trim/pred_for_slides/dataset_0/windows/group_partition_0/partition_0_e37f3d8a581b288ff047/layers/"

band_dropdown = widgets.Dropdown(
    options=list(band_groups.keys()),
    value="B05_B06_B07_B8A_B11_B12",
    description="Bands:"
)

slider = widgets.IntSlider(
    value=0,
    min=0,
    max=4,
    step=1,
    description='Image:',
    continuous_update=True
)

alpha_slider = widgets.FloatSlider(
    value=0.5,
    min=0.0,
    max=1.0,
    step=0.01,
    description='Alpha:',
    continuous_update=True
)


output = widgets.Output()

# Global States to be Rebuilt
images_norm = []
blended_images = []
alpha = 0.25

def load_band_group(group):
    global images_norm, blended_images

    paths = band_groups[group]
    images = []

    for rel in paths:
        rs_img = rs.open(base_path + rel)
        band_count = rs_img.count

        if band_count >= 3:
            arr = rs_img.read([3,2,1]).transpose((1,2,0))
        elif band_count == 2:
            b1 = rs_img.read(1)
            b2 = rs_img.read(2)
            arr = np.stack([b2, b1, b1], axis=-1)
        else:
            raise ValueError(f"Unsupported band count: {band_count}")

        images.append(arr)

    # normalize
    images_norm = []
    for img in images:
        s2_norm = equalize_hist(img.astype(np.float32))
        s2_norm = (s2_norm - s2_norm.min()) / (s2_norm.max() - s2_norm.min())
        images_norm.append(s2_norm)

    # resize overlay
    H, W, _ = images_norm[0].shape
    overlay_resized = resize(
        overlay,
        (H, W, 3),
        order=0,
        preserve_range=True,
        anti_aliasing=False
    )

    # blend
    rebuild_blended_images()

    # assign to globals
    globals()["images_norm"] = images_norm
    globals()["blended_images"] = blended_images


def update_slider(change):
    idx = change["new"]
    with output:
        clear_output(wait=True)
        plt.figure(figsize=(8, 8))
        plt.imshow(blended_images[idx])
        plt.title(f"Image {idx}")
        plt.axis("off")
        plt.show()

def rebuild_blended_images():
    global blended_images

    # resize overlay to match current images_norm
    H, W, _ = images_norm[0].shape
    overlay_resized = resize(
        overlay,
        (H, W, 3),
        order=0,
        preserve_range=True,
        anti_aliasing=False
    )

    blended_images = [0.5 * img + alpha * overlay_resized for img in images_norm]


def update_band(change):
    load_band_group(change["new"])   # rebuild blended_images

    idx = slider.value               # current slider position

    with output:
        clear_output(wait=True)
        plt.figure(figsize=(8, 8))
        plt.imshow(blended_images[idx])
        plt.title(f"{change['new']} — Image {idx}")
        plt.axis("off")
        plt.show()
        
def update_alpha(change):
    global alpha
    alpha = change["new"]

    rebuild_blended_images()

    # redraw current frame
    idx = slider.value
    with output:
        clear_output(wait=True)
        plt.figure(figsize=(8, 8))
        plt.imshow(blended_images[idx])
        plt.title(f"Image {idx} (alpha={alpha:.2f})")
        plt.axis("off")
        plt.show()


# Wire Callbacks 
slider.observe(update_slider, names="value")
band_dropdown.observe(update_band, names="value")
alpha_slider.observe(update_alpha, names="value")


# Initial Load
load_band_group(band_dropdown.value)
update_slider({"new": 0})


display(band_dropdown, slider, alpha_slider, output)



This cell loads a new set of prediction and ground‑truth rasters, computes TP/FP/FN masks, and builds a color‑coded overlay highlighting agreement and errors. It then provides an interactive viewer that lets you switch between different sensors and band groups while dynamically blending the overlay onto each image with adjustable transparency.

In [ ]:
# Load prediction and ground truth rasters
img_rs_pred_v2 = rs.open('/Users/juliansanders/Downloads/task_201f9420-249e-5945-b916-bfe6bc6ac10a/output/output/geotiff.tif')
img_pred_v2 = img_rs_pred_v2.read(1)

img_rs_gt_v2 = rs.open('/Users/juliansanders/Downloads/task_201f9420-249e-5945-b916-bfe6bc6ac10a/label/label/geotiff.tif')
img_gt_v2 = img_rs_gt_v2.read(1)

# Clip prediction to GT extent
starting_pos_v2 = img_rs_pred_v2.index(*img_rs_gt_v2.xy(0,0))
img_pred_clip_v2 = img_pred_v2[
    starting_pos_v2[0]:starting_pos_v2[0] + img_gt_v2.shape[0],
    starting_pos_v2[1]:starting_pos_v2[1] + img_gt_v2.shape[1]
]

# Build TP/FP/FN masks
gt_v2 = img_gt_v2.copy()
pred_v2 = img_pred_clip_v2.copy()

valid_mask_v2 = gt_v2 != 255
gt_bin_v2 = (gt_v2 == 1)
pred_bin_v2 = (pred_v2 == 1)

TP_v2 = (gt_bin_v2 & pred_bin_v2) & valid_mask_v2
FP_v2 = (~gt_bin_v2 & pred_bin_v2) & valid_mask_v2
FN_v2 = (gt_bin_v2 & ~pred_bin_v2) & valid_mask_v2

# RGB overlay
overlay_v2 = np.zeros((*gt_v2.shape, 3), dtype=np.float32)
overlay_v2[TP_v2] = [0, 1, 0]
overlay_v2[FP_v2] = [1, 0, 0]
overlay_v2[FN_v2] = [0, 0, 1]

overlay_mask_v2 = (overlay_v2.sum(axis=-1) > 0).astype(np.float32)[..., None]

# Sensor → band groups
base_v2 = "/Users/juliansanders/Downloads/task_201f9420-249e-5945-b916-bfe6bc6ac10a/"

sensor_paths_v2 = {
    "doveR": {
        "b01_b02_b03_b04": base_v2 + "doveR/b01_b02_b03_b04/geotiff.tif"
    },
    "sentinel2": {
        "B01_B09": base_v2 + "sentinel2_l2a_t0/B01_B09/geotiff.tif",
        "B02_B03_B04_B08": base_v2 + "sentinel2_l2a_t0/B02_B03_B04_B08/geotiff.tif",
        "B05_B06_B07_B8A_B11_B12": base_v2 + "sentinel2_l2a_t0/B05_B06_B07_B8A_B11_B12/geotiff.tif"
    },
    "superdove": {
        "b01_b02_b03_b04_b05_b06_b07_b08": base_v2 + "superdove_2years_after/b01_b02_b03_b04_b05_b06_b07_b08/geotiff.tif"
    }
}

# Widgets
sensor_dropdown_v2 = widgets.Dropdown(
    options=list(sensor_paths_v2.keys()),
    value="sentinel2",
    description="Sensor:"
)

band_dropdown_v2 = widgets.Dropdown(
    options=[],
    description="Bands:"
)

alpha_slider_v2 = widgets.FloatSlider(
    value=0.5,
    min=0.0,
    max=1.0,
    step=0.01,
    description="Overlay α:",
    continuous_update=True
)

output_v2 = widgets.Output()

# Global state
base_img_norm_v2 = None
alpha_v2 = 0.5

# Load and normalize selected band group
def load_band_image_v2(sensor, band_group):
    global base_img_norm_v2
    path = sensor_paths_v2[sensor][band_group]

    with rs.open(path) as src:
        band_count = src.count
        if band_count >= 3:
            arr = src.read([3,2,1]).transpose((1,2,0))
        elif band_count == 2:
            b1 = src.read(1)
            b2 = src.read(2)
            arr = np.stack([b2, b1, b1], axis=-1)
        else:
            b1 = src.read(1)
            arr = np.stack([b1, b1, b1], axis=-1)

    arr_norm = equalize_hist(arr.astype(np.float32))
    arr_norm = (arr_norm - arr_norm.min()) / (arr_norm.max() - arr_norm.min())
    base_img_norm_v2 = arr_norm

# Alpha compositing
def blend_image_v2():
    H, W, _ = base_img_norm_v2.shape

    overlay_resized_v2 = resize(
        overlay_v2, (H, W, 3),
        order=0, preserve_range=True, anti_aliasing=False
    )

    mask_resized_v2 = resize(
        overlay_mask_v2, (H, W, 1),
        order=0, preserve_range=True, anti_aliasing=False
    )

    return (
        base_img_norm_v2 * (1 - alpha_v2 * mask_resized_v2) +
        overlay_resized_v2 * (alpha_v2 * mask_resized_v2)
    )

# Callbacks
def update_sensor_v2(change):
    sensor = change["new"]
    band_dropdown_v2.options = list(sensor_paths_v2[sensor].keys())
    band_dropdown_v2.value = list(sensor_paths_v2[sensor].keys())[0]

def update_band_v2(change):
    load_band_image_v2(sensor_dropdown_v2.value, change["new"])
    update_display_v2()

def update_alpha_v2(change):
    global alpha_v2
    alpha_v2 = change["new"]
    update_display_v2()

def update_display_v2():
    blended = blend_image_v2()
    with output_v2:
        clear_output(wait=True)
        plt.figure(figsize=(8,8))
        plt.imshow(blended)
        plt.axis("off")
        plt.show()

# Connect widgets
sensor_dropdown_v2.observe(update_sensor_v2, names="value")
band_dropdown_v2.observe(update_band_v2, names="value")
alpha_slider_v2.observe(update_alpha_v2, names="value")

# Initial load
update_sensor_v2({"new": sensor_dropdown_v2.value})
load_band_image_v2(sensor_dropdown_v2.value, band_dropdown_v2.value)
update_display_v2()

display(sensor_dropdown_v2, band_dropdown_v2, alpha_slider_v2, output_v2)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from skimage.transform import resize
from skimage.exposure import equalize_hist
import rasterio as rs
from sarpy.visualization.remap import density

# Load prediction + ground truth for task 1

task_id_t1 = "8c602997-b95d-529d-98a9-6ed9899e6f58"

pred_path_t1 = f"/Users/juliansanders/Downloads/data 3/datasets/olmo/defid2/oerun_cluster_30d_trim/pred_for_slides2/task_{task_id_t1}/output/output/geotiff.tif"
gt_path_t1   = f"/Users/juliansanders/Downloads/data 3/datasets/olmo/defid2/oerun_cluster_30d_trim/pred_for_slides2/task_{task_id_t1}/label/label/geotiff.tif"
sd_path_t1   = f"/Users/juliansanders/Downloads/data 3/datasets/olmo/defid2/oerun_cluster_30d_trim/pred_for_slides2/task_{task_id_t1}/superdove_mar_sept_2021_random/b01_b02_b03_b04_b05_b06_b07_b08/geotiff.tif"

img_rs_pred_t1 = rs.open(pred_path_t1)
img_pred_t1 = img_rs_pred_t1.read(1)

img_rs_gt_t1 = rs.open(gt_path_t1)
img_gt_t1 = img_rs_gt_t1.read(1)

# Align prediction to GT extent
start_t1 = img_rs_pred_t1.index(*img_rs_gt_t1.xy(0, 0))
img_pred_clip_t1 = img_pred_t1[
    start_t1[0]:start_t1[0] + img_gt_t1.shape[0],
    start_t1[1]:start_t1[1] + img_gt_t1.shape[1]
]

# Build TP / FP / FN masks

gt_bin_t1 = (img_gt_t1 == 1)
pred_bin_t1 = (img_pred_clip_t1 == 1)
valid_mask_t1 = (img_gt_t1 != 255)

TP_t1 = (gt_bin_t1 & pred_bin_t1) & valid_mask_t1
FP_t1 = (~gt_bin_t1 & pred_bin_t1) & valid_mask_t1
FN_t1 = (gt_bin_t1 & ~pred_bin_t1) & valid_mask_t1

overlay_t1 = np.zeros((*img_gt_t1.shape, 3), dtype=np.float32)
overlay_t1[TP_t1] = [0, 1, 0]
overlay_t1[FP_t1] = [1, 0, 0]
overlay_t1[FN_t1] = [0, 0, 1]

overlay_mask_t1 = (overlay_t1.sum(axis=-1) > 0).astype(np.float32)[..., None]

# Load SuperDove RGB (bands 6,4,2)

img_rs_sd_t1 = rs.open(sd_path_t1)
base_img_t1 = img_rs_sd_t1.read([6, 4, 2]).transpose((1, 2, 0)).astype(np.float32)

# Normalize with density()
base_norm_t1 = density(base_img_t1) / 255

# Resize overlay to match SuperDove
H_t1, W_t1, _ = base_norm_t1.shape

overlay_resized_t1 = resize(
    overlay_t1, (H_t1, W_t1, 3),
    order=0, preserve_range=True, anti_aliasing=False
)

mask_resized_t1 = resize(
    overlay_mask_t1, (H_t1, W_t1, 1),
    order=0, preserve_range=True, anti_aliasing=False
)

# Alpha blend

alpha_t1 = 0.35

blended_t1 = (
    base_norm_t1 * (1 - alpha_t1 * mask_resized_t1) +
    overlay_resized_t1 * (alpha_t1 * mask_resized_t1)
)

# Side-by-side plot

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# Left: TP/FP/FN shaded overlay
axes[0].imshow(blended_t1)
axes[0].set_title("TP / FP / FN Overlay")
axes[0].axis("off")

# Right: Teal contour around predictions
axes[1].imshow(base_norm_t1)
axes[1].contour(
    pred_bin_t1,
    levels=[0.5],
    colors="#008080",
    linewidths=4
)
axes[1].set_title("Prediction Contour (Teal)")
axes[1].axis("off")

plt.tight_layout()
plt.show()
